# stable-diffusion.cpp sd-server on Colab

在 Colab GPU 上运行 [stable-diffusion.cpp](https://github.com/leejet/stable-diffusion.cpp) 的 sd-server，供 ComfyUI-YogurtNodes 的 **stable-diffusion.cpp** 节点远程调用。

流程：从官方镜像 `ghcr.io/leejet/stable-diffusion.cpp:master-cuda` 下载预编译的 sd-server（约 270MB，无需编译）→ 下载模型 → 启动 sd-server → 在前面加一层 token 鉴权代理 → 用 cloudflared 临时隧道暴露到公网 → 输出 Provider 节点要填的 `base_url` 和 `api_key`。

**使用前请注意**

- 菜单 *Runtime → Change runtime type* 选择 GPU。推荐 **L4**：镜像自带 sm89 机器码，开箱即跑。T4（sm75）、A100（sm80）只有 PTX，第一次生成时驱动要现场编译 CUDA kernel，会额外等待一段时间；编译结果缓存到 Google Drive 后，后续会话不再重复。GPU 型号越好，每小时消耗的计算单元越多。
- Colab FAQ 规定：免费额度下禁止“绕过 notebook 主要通过 Web UI 交互”；计算单元余额为正时这条限制解除。但“与 Colab 交互式计算无关的 Web 服务”和“连接远程代理”对所有运行时都禁止。本 notebook 只供你自己的 ComfyUI 调用，属于灰色地带，运行时可能被终止。
- 运行时空闲会断开。请保持本页打开；用完后选择 *Runtime → Disconnect and delete runtime*，停止消耗计算单元。
- 隧道地址和 token 每次启动都会变化，需要重新填到 Provider 节点。


In [ ]:
#@title 1. 参数 { display-mode: "form" }
#@markdown 模型直链（默认 SD 1.5，约 4.3GB，和 YogurtNodes 测试工作流的默认 512 尺寸、20 步匹配）。
MODEL_URL = "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"  #@param {type:"string"}
#@markdown 下载受限模型（gated）时需要 Hugging Face token。
HF_TOKEN = ""  #@param {type:"string"}
#@markdown 额外的 sd-server 参数，例如 `--offload-to-cpu`、`--vae ...`。
SERVER_ARGS = ""  #@param {type:"string"}
#@markdown 官方镜像 tag（`master-cuda` 随上游 master 更新）。
SDCPP_IMAGE_TAG = "master-cuda"  #@param {type:"string"}
#@markdown 把模型和 CUDA JIT 缓存存到 Google Drive（推荐，否则每次会话都要重新下载模型、T4/A100 还要重新 JIT）。
USE_DRIVE_CACHE = True  #@param {type:"boolean"}
#@markdown 留空则每次随机生成。
API_TOKEN = ""  #@param {type:"string"}

import secrets

API_TOKEN = API_TOKEN.strip() or secrets.token_urlsafe(24)
SERVER_PORT, PROXY_PORT = 7870, 7871
print("参数已设置")


In [ ]:
#@title 2. 下载预编译 sd-server（官方 CUDA 镜像，无需编译）
"""从官方镜像 ghcr.io/leejet/stable-diffusion.cpp:master-cuda 下载预编译的 sd-server（只取 /sd.cpp/bin 这一层），
并为它找齐运行所需的 CUDA 动态库。镜像基于 Ubuntu 24.04 + CUDA 12，需要 glibc ≥ 2.38、libcublas.so.12、libnccl.so.2。
"""

import glob
import json
import os
import subprocess
import tarfile
import urllib.request
from pathlib import Path

IMAGE_REPO = "leejet/stable-diffusion.cpp"
_ACCEPT = ", ".join([
    "application/vnd.oci.image.index.v1+json", "application/vnd.docker.distribution.manifest.list.v2+json",
    "application/vnd.oci.image.manifest.v1+json", "application/vnd.docker.distribution.manifest.v2+json",
])


def _registry(path, token, accept=_ACCEPT):
    request = urllib.request.Request(f"https://ghcr.io/v2/{IMAGE_REPO}/{path}",
                                     headers={"Authorization": f"Bearer {token}", "Accept": accept})
    return urllib.request.urlopen(request, timeout=60)


def fetch_sd_server(dest, tag="master-cuda"):
    """下载镜像中 /sd.cpp/bin 所在的层并解压到 dest，返回 (sd-server 路径, 镜像构建时间)。"""
    token_url = f"https://ghcr.io/token?scope=repository:{IMAGE_REPO}:pull"
    token = json.load(urllib.request.urlopen(token_url, timeout=60))["token"]
    manifest = json.load(_registry(f"manifests/{tag}", token))
    if "manifests" in manifest:
        entry = next(m for m in manifest["manifests"] if m.get("platform", {}).get("architecture") == "amd64")
        manifest = json.load(_registry(f"manifests/{entry['digest']}", token))
    config = json.load(_registry(f"blobs/{manifest['config']['digest']}", token, accept="*/*"))
    steps = [h.get("created_by", "") for h in config.get("history", []) if not h.get("empty_layer")]
    layer = next(layer for layer, step in zip(manifest["layers"], steps) if "/sd.cpp/build/bin" in step)

    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    print(f"下载 {tag} 的 bin 层 {layer['size'] / 1048576:.0f} MB（镜像构建于 {config.get('created', '?')[:19]}）", flush=True)
    with _registry(f"blobs/{layer['digest']}", token, accept="*/*") as response, \
            tarfile.open(fileobj=response, mode="r|gz") as tar:
        for member in tar:
            name = member.name.lstrip("./")
            if not name.startswith("sd.cpp/bin/") or member.isdir():
                continue
            target = dest / Path(name).name
            if target.is_symlink() or target.exists():
                target.unlink()
            if member.issym():
                target.symlink_to(member.linkname)
            elif member.isfile():
                with tar.extractfile(member) as source, open(target, "wb") as out:
                    while chunk := source.read(1 << 20):
                        out.write(chunk)
                target.chmod(0o755)
    return dest / "sd-server", config.get("created", "")


def _missing(binaries, lib_path):
    env = {**os.environ, "LD_LIBRARY_PATH": lib_path}
    missing = set()
    for binary in binaries:
        result = subprocess.run(["ldd", str(binary)], capture_output=True, text=True, env=env)
        if result.returncode != 0:
            raise RuntimeError(f"ldd {binary} 失败：{(result.stderr or result.stdout).strip()}")
        missing.update(line.split()[0] for line in result.stdout.splitlines() if "not found" in line)
    return missing


def library_path(bin_dir, search_dirs=()):
    """返回能让 sd-server 及 CUDA 后端找齐依赖的 LD_LIBRARY_PATH；找不齐时报错并列出缺失的库。"""
    bin_dir = Path(bin_dir)
    binaries = [bin_dir / "sd-server", bin_dir / "libggml-cuda.so"]
    dirs = [str(bin_dir)] + [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d]
    missing = _missing(binaries, ":".join(dirs))
    for soname in sorted(missing):
        for pattern in search_dirs:
            hits = glob.glob(os.path.join(pattern, soname))
            if hits:
                dirs.append(os.path.dirname(hits[0]))
                break
    lib_path = ":".join(dict.fromkeys(dirs))
    still_missing = _missing(binaries, lib_path)
    if still_missing:
        raise RuntimeError("sd-server 缺少动态库：" + ", ".join(sorted(still_missing)))
    return lib_path


import site

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("没有检测到 GPU：Runtime → Change runtime type 选择 GPU 后重新运行")
GPU_NAME, COMPUTE_CAP, GPU_MEMORY = [part.strip() for part in gpu.stdout.strip().splitlines()[0].split(",")]
ARCH = COMPUTE_CAP.replace(".", "")

SD_SERVER, IMAGE_CREATED = fetch_sd_server("/content/sdcpp/bin", SDCPP_IMAGE_TAG)
# Colab 自带 CUDA 12 的 cublas / cudart / nccl；缺失时再到 torch 附带的 nvidia/*/lib 里找
search_dirs = [os.path.join(path, "nvidia", "*", "lib") for path in site.getsitepackages()]
search_dirs += ["/usr/local/cuda/lib64", "/usr/lib64-nvidia", "/usr/lib/x86_64-linux-gnu"]
SDCPP_LD_PATH = library_path("/content/sdcpp/bin", search_dirs)
jit_note = "自带机器码" if ARCH in ("86", "89") else "首次生成需要驱动 JIT 编译"
print(f"GPU: {GPU_NAME} ({GPU_MEMORY}, sm{ARCH}, {jit_note})；sd-server 镜像构建于 {IMAGE_CREATED[:19]}")


In [ ]:
#@title 3. 下载模型
import subprocess
from pathlib import Path
from urllib.parse import urlparse

if USE_DRIVE_CACHE:
    from google.colab import drive

    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/sdcpp-colab")
else:
    CACHE = Path("/content/sdcpp-cache")
CACHE.mkdir(parents=True, exist_ok=True)

MODELS = CACHE / "models"
MODELS.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS / Path(urlparse(MODEL_URL).path).name
if MODEL_PATH.exists():
    print("已存在", MODEL_PATH)
else:
    partial = MODEL_PATH.with_name(MODEL_PATH.name + ".part")
    cmd = ["curl", "-L", "--fail", "-C", "-", "-o", str(partial), MODEL_URL]
    if HF_TOKEN:
        cmd[1:1] = ["-H", f"Authorization: Bearer {HF_TOKEN}"]
    print("下载", MODEL_URL, flush=True)
    subprocess.run(cmd, check=True)
    partial.rename(MODEL_PATH)
print(f"{MODEL_PATH}  {MODEL_PATH.stat().st_size / 1024 ** 3:.2f} GB")


In [ ]:
#@title 4. 写入鉴权代理脚本
from pathlib import Path

AUTH_PROXY_SOURCE = r'''"""sd-server 前的 Bearer token 鉴权代理（仅用标准库）。

sd-server 本身没有鉴权；隧道把端口暴露到公网后，只有带 `Authorization: Bearer <token>` 的请求才会被转发。
环境变量：SDCPP_TOKEN（必填）、SDCPP_UPSTREAM（默认 http://127.0.0.1:7870）、SDCPP_PROXY_PORT（默认 7871）。
"""

import hmac
import json
import os
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

UPSTREAM = os.environ.get("SDCPP_UPSTREAM", "http://127.0.0.1:7870").rstrip("/")
PORT = int(os.environ.get("SDCPP_PROXY_PORT", "7871"))
EXPECTED = f"Bearer {os.environ['SDCPP_TOKEN']}".encode()


class Proxy(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def _forward(self):
        if not hmac.compare_digest(self.headers.get("Authorization", "").encode(), EXPECTED):
            return self._reply(401, b'{"error":"unauthorized"}', "application/json")
        length = int(self.headers.get("Content-Length") or 0)
        request = urllib.request.Request(
            UPSTREAM + self.path, data=self.rfile.read(length) if length else None, method=self.command
        )
        if self.headers.get("Content-Type"):
            request.add_header("Content-Type", self.headers["Content-Type"])
        try:
            with urllib.request.urlopen(request, timeout=600) as response:
                self._reply(response.status, response.read(), response.headers.get("Content-Type", "application/json"))
        except urllib.error.HTTPError as error:
            self._reply(error.code, error.read(), error.headers.get("Content-Type", "application/json"))
        except urllib.error.URLError as error:
            body = json.dumps({"error": f"sd-server unreachable: {error.reason}"}).encode()
            self._reply(502, body, "application/json")

    def _reply(self, status, body, content_type):
        self.send_response(status)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    do_GET = do_POST = _forward

    def log_message(self, fmt, *args):
        pass


if __name__ == "__main__":
    print(f"auth proxy :{PORT} -> {UPSTREAM}", flush=True)
    ThreadingHTTPServer(("127.0.0.1", PORT), Proxy).serve_forever()
'''
Path('/content/auth_proxy.py').write_text(AUTH_PROXY_SOURCE)
print('已写入 /content/auth_proxy.py')


In [ ]:
#@title 5. 启动 sd-server + 鉴权代理 + 公网隧道
import json
import os
import re
import shlex
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

# 重复运行本单元时先停掉上一次启动的进程
for _name in ("sd_server_proc", "proxy_proc", "tunnel_proc"):
    _proc = globals().get(_name)
    if _proc is not None and _proc.poll() is None:
        _proc.terminate()
        _proc.wait(15)


def http_status(url, token=None):
    headers = {"User-Agent": "sdcpp-colab-check"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=15) as response:
            return response.status, response.read()
    except urllib.error.HTTPError as error:
        return error.code, error.read()
    except (urllib.error.URLError, OSError):
        return None, b""


def wait_for(check, proc, log_path, timeout, what):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"{what} 已退出（exit {proc.returncode}），日志末尾：\n"
                               + Path(log_path).read_text(errors="replace")[-3000:])
        result = check()
        if result:
            return result
        time.sleep(2)
    raise TimeoutError(f"{what} 在 {timeout}s 内未就绪，日志末尾：\n" + Path(log_path).read_text(errors="replace")[-3000:])


SERVER_LOG, PROXY_LOG, TUNNEL_LOG = "/content/sd-server.log", "/content/auth_proxy.log", "/content/cloudflared.log"

# T4 / A100 的 CUDA kernel 由驱动 JIT 编译，结果缓存到 CACHE（Drive）供后续会话复用
server_env = {**os.environ, "LD_LIBRARY_PATH": SDCPP_LD_PATH,
              "CUDA_CACHE_PATH": str(CACHE / f"cuda-jit-sm{ARCH}"), "CUDA_CACHE_MAXSIZE": str(4 * 1024 ** 3)}
sd_server_proc = subprocess.Popen(
    [str(SD_SERVER), "-m", str(MODEL_PATH), "--listen-ip", "127.0.0.1", "--listen-port", str(SERVER_PORT),
     *shlex.split(SERVER_ARGS)],
    env=server_env, stdout=open(SERVER_LOG, "w"), stderr=subprocess.STDOUT,
)
print("加载模型中……", flush=True)
wait_for(lambda: http_status(f"http://127.0.0.1:{SERVER_PORT}/sdcpp/v1/capabilities")[0] == 200,
         sd_server_proc, SERVER_LOG, 900, "sd-server")

proxy_proc = subprocess.Popen(
    [sys.executable, "/content/auth_proxy.py"],
    env={**os.environ, "SDCPP_TOKEN": API_TOKEN, "SDCPP_UPSTREAM": f"http://127.0.0.1:{SERVER_PORT}",
         "SDCPP_PROXY_PORT": str(PROXY_PORT)},
    stdout=open(PROXY_LOG, "w"), stderr=subprocess.STDOUT,
)
wait_for(lambda: http_status(f"http://127.0.0.1:{PROXY_PORT}/sdcpp/v1/capabilities", API_TOKEN)[0] == 200,
         proxy_proc, PROXY_LOG, 30, "鉴权代理")

CLOUDFLARED = Path("/content/cloudflared")
if not CLOUDFLARED.exists():
    subprocess.run(["curl", "-L", "--fail", "-o", str(CLOUDFLARED),
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    CLOUDFLARED.chmod(0o755)
tunnel_proc = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PROXY_PORT}"],
    stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT,
)


def tunnel_url():
    match = re.search(r"https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com", Path(TUNNEL_LOG).read_text(errors="replace"))
    return match.group(0) if match else None


PUBLIC_URL = wait_for(tunnel_url, tunnel_proc, TUNNEL_LOG, 120, "cloudflared")
print("等待公网地址生效……", flush=True)
wait_for(lambda: http_status(f"{PUBLIC_URL}/sdcpp/v1/capabilities", API_TOKEN)[0] == 200,
         tunnel_proc, TUNNEL_LOG, 180, "公网地址")
no_token_status = http_status(f"{PUBLIC_URL}/sdcpp/v1/capabilities")[0]
capabilities = json.loads(http_status(f"http://127.0.0.1:{SERVER_PORT}/sdcpp/v1/capabilities")[1])

print("=" * 72)
print("在 ComfyUI 的「Provider: stable-diffusion.cpp」节点填写：")
print(f"  base_url = {PUBLIC_URL}")
print(f"  api_key  = {API_TOKEN}")
print("=" * 72)
print(f"模型：{capabilities.get('model', {}).get('name')}   GPU：{GPU_NAME}")
print(f"自检：带 token → HTTP 200；不带 token → HTTP {no_token_status}（应为 401）")
print("保持本页打开；用完请 Runtime → Disconnect and delete runtime。")


In [ ]:
#@title 6. 查看状态和日志（随时可运行）
from pathlib import Path

for _name in ("sd_server_proc", "proxy_proc", "tunnel_proc"):
    _proc = globals().get(_name)
    state = "未启动" if _proc is None else ("运行中" if _proc.poll() is None else f"已退出（exit {_proc.returncode}）")
    print(f"{_name:16s} {state}")
for _log in ("/content/sd-server.log", "/content/auth_proxy.log", "/content/cloudflared.log"):
    if Path(_log).exists():
        print(f"\n----- {_log}（末尾）-----")
        print(Path(_log).read_text(errors="replace")[-1500:])


In [ ]:
#@title 7. 停止服务（不会释放运行时；要停止计费请 Runtime → Disconnect and delete runtime）
for _name in ("tunnel_proc", "proxy_proc", "sd_server_proc"):
    _proc = globals().get(_name)
    if _proc is not None and _proc.poll() is None:
        _proc.terminate()
        _proc.wait(15)
        print("已停止", _name)
